# 02 — Modeling & Recommendation Walkthrough

This notebook tells the *methodology story* the EDA notebook doesn't:

1. **Vectorization choice** — why a TF-IDF on canonical-skill bags (instead of raw description text) gives a more explainable Match Score.
2. **Cosine similarity** — what the Match Score is mathematically, and why values cluster where they do.
3. **Skill-gap analysis** — set ops and weighted coverage worked through on a real example.
4. **Skill graph** — co-occurrence (NPMI) inspection.
5. **Bridge-role pathing** — finding shortest skill-jump paths between roles.

Run [01_eda.ipynb](01_eda.ipynb) first if you haven't preprocessed the data yet.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.recommender import CareerRecommender
rec = CareerRecommender.load_or_build()
print(f'Roles: {len(rec.roles)},  Postings: {len(rec.postings):,},  TF-IDF vocab: {rec.tfidf.vocabulary_size()}')

## 1. Vectorization choice

Each role is represented as a *document* of canonical skill tokens, with each skill repeated proportionally to the fraction of postings for that role that mention it (capped at 10). TF-IDF then up-weights skills that are characteristic of a role and down-weights universal skills like Excel.

Compare a few roles' top TF-IDF features:

In [ ]:
vocab = np.array(rec.tfidf.vectorizer.get_feature_names_out())
matrix = rec.tfidf.role_matrix.toarray()

def top_tfidf_for(title_substring: str, k: int = 10):
    matches = rec.roles[rec.roles['title'].str.contains(title_substring, case=False)]
    if matches.empty: return None
    idx = matches.index[0]
    weights = matrix[idx]
    top_idx = np.argsort(-weights)[:k]
    return pd.DataFrame({'skill': vocab[top_idx], 'tfidf': weights[top_idx].round(3)})

for t in ['Software Engineer', 'Data Scientist', 'Project Manager', 'Sales Manager']:
    print(f'\n=== Top skills for: {t} ===')
    print(top_tfidf_for(t))

## 2. Cosine similarity — what the Match Score really is

Match Score = cos(user_vector, role_vector) ∈ [0, 1].

Let's see the full distribution of scores against every role for one user profile.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
user_skills = {'Python', 'SQL', 'Pandas', 'Statistics', 'Machine Learning'}
known = [s for s in user_skills if s in rec.tfidf.vectorizer.vocabulary_]
qv = rec.tfidf.vectorizer.transform([known])
sims = cosine_similarity(qv, rec.tfidf.role_matrix).ravel()

plt.figure(figsize=(8, 3))
plt.hist(sims, bins=50, edgecolor='white')
plt.title(f'Match-score distribution across all {len(sims)} roles\nuser_skills = {sorted(user_skills)}')
plt.xlabel('Cosine similarity'); plt.ylabel('# roles'); plt.tight_layout(); plt.show()

print(f'Top 10 roles for this user:')
for i in np.argsort(-sims)[:10]:
    print(f'  {sims[i]:.3f}  {rec.roles.iloc[i]["title"]}')

## 3. Skill-gap analysis on a real example

In [ ]:
_, recs = rec.recommend('Python, SQL, Pandas, Statistics, Machine Learning', top_k=3)
for r in recs:
    print(f'\n>>> {r.title}  (score={r.match_score:.3f}, n_postings={r.n_postings})')
    print(f'    weighted coverage: {r.gap.weighted_coverage:.2%}')
    print(f'    matched ({len(r.gap.matched)}): {r.gap.matched}')
    print(f'    missing top-5: {[s for s,_ in r.gap.missing_ranked[:5]]}')
    print(f'    extras: {r.gap.extras}')

## 4. Skill co-occurrence graph

In [ ]:
from src.skill_graph import adjacent_skills
for s in ['Python', 'AWS', 'React', 'Pandas', 'Docker', 'TensorFlow']:
    print(f'\n{s} → strongest co-occurring skills:')
    for n, w in adjacent_skills(rec.skill_graph, s, top_k=8):
        print(f'  {w:5.2f}  {n}')

## 5. Bridge-role pathing

Suppose the user is currently aligned to a Data Analyst-like role and wants to move to a Machine Learning Engineer role. The role-transition graph gives the shortest sequence of intermediate roles.

In [ ]:
user = {'Python', 'SQL', 'Excel', 'Statistics'}
src_role = rec.closest_role_for_user(user)
print('Closest current role:', src_role)

# Pick a target ML-leaning role
candidates = rec.roles[rec.roles['title'].str.contains('Machine Learning|Data Scientist', case=False, regex=True)]
print('\nCandidate target roles:')
print(candidates[['role_id','title','n_postings']].head(5).to_string(index=False))

In [ ]:
# Try bridging to the first one with at least one path
for target in candidates['role_id'].head(5):
    path = rec.bridge_path(user, target, max_hops=3)
    if path:
        print(f'Bridge to: {target}')
        for step in path:
            print(f"  {step['from_title']} → {step['to_title']}  (learn: {step['new_skills']})")
        break
else:
    print('No path within 3 hops — try increasing max_hops or relaxing role-graph thresholds.')